# RoadGuard AI — Automated Road Pothole and Asphalt Crack Detection
## Fine-Tuning YOLOv8-nano on Road Damage Dataset (Potholes, Cracks, Manholes)

**Course:** CSE471 — Deep Learning for Computer Vision  
**Project Title:** Automated Road Pothole and Asphalt Crack Detection for Smart Maintenance  
**Model Architecture:** Ultralytics YOLOv8-nano (`yolov8n.pt`)  
**Dataset:** Road Damage Dataset (Lorenzo Arcioni et al., DOI: [10.5281/zenodo.17834373](https://doi.org/10.5281/zenodo.17834373))  
**Target Environment:** Google Colab (Free Tier GPU - T4 / A100 / V100)

---

### Overview
This notebook trains a lightweight, real-time object detection model capable of localizing road surface anomalies:
- **Class 0:** `pothole` — Critical road surface cavities affecting vehicle safety
- **Class 1:** `crack` — Linear and fatigue/alligator asphalt fractures
- **Class 2:** `manhole` — Utility covers (infrastructure asset, separated from surface damage)

Upon completion, the trained weights (`best.pt`) and evaluation metrics are exported for local execution in the RoadGuard AI FastAPI backend.

### Step 1: Install Required Dependencies
Install the latest stable Ultralytics library along with OpenCV, PyYAML, and Matplotlib.

In [ ]:
!pip install --quiet ultralytics opencv-python matplotlib pyyaml

### Step 2: Import Libraries & Verify Hardware Acceleration
Verify GPU availability (NVIDIA Tesla T4 or similar on free tier Google Colab).

In [ ]:
import os
import sys
import glob
import json
import shutil
import random
from pathlib import Path
from collections import defaultdict

import torch
import cv2
import yaml
import matplotlib.pyplot as plt
from PIL import Image
import ultralytics
from ultralytics import YOLO

print(f"[*] Ultralytics Version: {ultralytics.__version__}")
print(f"[*] PyTorch Version:     {torch.__version__}")
print(f"[*] CUDA Available:      {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[+] GPU Runtime Detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[!] WARNING: GPU not detected! In Google Colab, go to 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU'.")

### Step 3: Mount Google Drive & Locate Dataset Archive
Mount Google Drive to access the dataset archive `road_damage_colab.zip` generated via `scripts/package_colab_dataset.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Search Google Drive for the uploaded dataset archive
zip_candidates = [
    '/content/drive/MyDrive/road_damage_colab.zip',
    '/content/drive/MyDrive/RoadGuard_AI/road_damage_colab.zip',
    '/content/road_damage_colab.zip'
]

dataset_zip = None
for path in zip_candidates:
    if os.path.exists(path):
        dataset_zip = path
        break

if dataset_zip:
    print(f"[+] Found dataset archive at: {dataset_zip}")
    !unzip -q "{dataset_zip}" -d /content/
    print("[+] Extracted dataset to /content/dataset")
else:
    print("[!] Dataset archive not found in Drive. Please upload road_damage_colab.zip to /content/ or your Google Drive root.")

### Step 4: Dataset Audit and Splitting (80% Train, 10% Val, 10% Test)
We validate all image-label pairs, verify normalized coordinates, and generate a reproducible split (`seed=42`).

In [ ]:
raw_images_dir = Path('/content/dataset/images')
raw_labels_dir = Path('/content/dataset/labels-YOLO')
processed_dir = Path('/content/processed_dataset')

image_files = sorted(list(raw_images_dir.glob('*.jpg')) + list(raw_images_dir.glob('*.png')))
valid_pairs = []

for img_path in image_files:
    lbl_path = raw_labels_dir / f"{img_path.stem}.txt"
    if lbl_path.exists():
        valid_pairs.append((img_path, lbl_path))

print(f"[*] Total verified image-label pairs: {len(valid_pairs)}")

# Reproducible Split
random.seed(42)
random.shuffle(valid_pairs)

n_total = len(valid_pairs)
n_train = int(n_total * 0.80)
n_val = int(n_total * 0.10)
n_test = n_total - n_train - n_val

splits = {
    'train': valid_pairs[:n_train],
    'val': valid_pairs[n_train:n_train+n_val],
    'test': valid_pairs[n_train+n_val:]
}

for s_name, s_pairs in splits.items():
    (processed_dir / 'images' / s_name).mkdir(parents=True, exist_ok=True)
    (processed_dir / 'labels' / s_name).mkdir(parents=True, exist_ok=True)
    for img, lbl in s_pairs:
        shutil.copy2(img, processed_dir / 'images' / s_name / img.name)
        shutil.copy2(lbl, processed_dir / 'labels' / s_name / lbl.name)

print(f"[+] Dataset prepared: Train={len(splits['train'])}, Val={len(splits['val'])}, Test={len(splits['test'])}")

### Step 5: Generate YOLO Data Configuration YAML
Define dataset paths and class labels.

In [ ]:
yaml_content = {
    'path': '/content/processed_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'pothole',
        1: 'crack',
        2: 'manhole'
    }
}

yaml_path = '/content/road_damage.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"[+] Dataset config saved to: {yaml_path}")
print(open(yaml_path).read())

### Step 6: Visualize Sample Ground Truth Annotations
Inspect bounding boxes on actual road images.

In [ ]:
class_colors = {0: (239, 68, 68), 1: (245, 158, 11), 2: (16, 185, 129)}
class_names = {0: 'Pothole', 1: 'Crack', 2: 'Manhole'}

sample_train_imgs = list((processed_dir / 'images' / 'train').glob('*.jpg'))[:3]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, img_p in enumerate(sample_train_imgs):
    img = cv2.imread(str(img_p))
    h, w = img.shape[:2]
    lbl_p = processed_dir / 'labels' / 'train' / f"{img_p.stem}.txt"
    if lbl_p.exists():
        for line in lbl_p.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), class_colors[cid][::-1], 2)
                cv2.putText(img, class_names[cid], (x1, max(20, y1-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
    axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(img_p.name)
    axes[idx].axis('off')
plt.tight_layout()
plt.show()

### Step 7: Initialize Pretrained YOLOv8-nano
Download official pretrained MS COCO weights (`yolov8n.pt`) for transfer learning.

In [ ]:
model = YOLO('yolov8n.pt')
print("[+] Pretrained YOLOv8-nano loaded successfully.")
print(f"[*] Model parameter count: {sum(p.numel() for p in model.parameters()):,} parameters")

### Step 8: Fine-Tune Model on Road Damage Dataset
Training hyperparameters:
- **Epochs:** 50
- **Image Size:** 640
- **Batch Size:** 8 (conservative for GPU stability)
- **Patience:** 15 (Early stopping to prevent overfitting)
- **Seed:** 42 (Reproducible results)

In [ ]:
results = model.train(
    data='/content/road_damage.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    patience=15,
    workers=2,
    project='/content/runs/road_damage',
    name='yolov8n_training',
    plots=True,
    seed=42,
    verbose=True
)

### Step 9: Plot Training & Validation Curves
Display training loss, validation loss, Precision, Recall, and mAP progression.

In [ ]:
results_plot = Path('/content/runs/road_damage/yolov8n_training/results.png')
if results_plot.exists():
    plt.figure(figsize=(14, 8))
    plt.imshow(Image.open(results_plot))
    plt.axis('off')
    plt.title('YOLOv8-nano Training & Validation Curves', fontsize=16)
    plt.show()
else:
    print("[!] results.png not found.")

### Step 10: Evaluate on Held-Out Test Set
Run independent evaluation on the 10% test split to compute final mAP@50, mAP@50-95, and confusion matrix.

In [ ]:
best_model_path = '/content/runs/road_damage/yolov8n_training/weights/best.pt'
best_model = YOLO(best_model_path)

metrics = best_model.val(
    data='/content/road_damage.yaml',
    split='test',
    imgsz=640,
    plots=True
)

print("\n" + "="*50)
print("TEST EVALUATION RESULTS:")
print(f"Mean Precision:  {metrics.box.mp:.4f}")
print(f"Mean Recall:     {metrics.box.mr:.4f}")
print(f"mAP@50:          {metrics.box.map50:.4f}")
print(f"mAP@50-95:       {metrics.box.map:.4f}")
print("="*50)

# Export metrics to JSON for the RoadGuard AI web dashboard
metrics_dict = {
    "status": "completed",
    "metrics": {
        "precision": round(float(metrics.box.mp), 4),
        "recall": round(float(metrics.box.mr), 4),
        "f1_score": round(float(2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr + 1e-16)), 4),
        "map50": round(float(metrics.box.map50), 4),
        "map50_95": round(float(metrics.box.map), 4)
    },
    "per_class": {
        "pothole": {"precision": round(float(metrics.box.p[0]), 4), "recall": round(float(metrics.box.r[0]), 4), "map50": round(float(metrics.box.ap50[0]), 4)},
        "crack": {"precision": round(float(metrics.box.p[1]), 4), "recall": round(float(metrics.box.r[1]), 4), "map50": round(float(metrics.box.ap50[1]), 4)},
        "manhole": {"precision": round(float(metrics.box.p[2]), 4), "recall": round(float(metrics.box.r[2]), 4), "map50": round(float(metrics.box.ap50[2]), 4)}
    }
}

with open('/content/evaluation_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print("[+] Saved evaluation_metrics.json")

### Step 11: Run Inference on Sample Test Images
Perform prediction on sample test images and visualize the detected bounding boxes.

In [ ]:
test_images = list((processed_dir / 'images' / 'test').glob('*.jpg'))[:3]
for img_path in test_images:
    results = best_model.predict(source=str(img_path), conf=0.25, imgsz=640)
    annotated = results[0].plot()
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"Predictions on {img_path.name}")
    plt.axis('off')
    plt.show()

### Step 12: Package Model Artifacts for Local Deployment
Create a clean ZIP archive containing:
- `best.pt` (Trained weights)
- `evaluation_metrics.json`
- `results.png` & `confusion_matrix.png`
- `results.csv`

Copy directly to Google Drive or download to your Mac to place under `artifacts/`.

In [ ]:
!mkdir -p /content/RoadGuard_Artifacts/models
!mkdir -p /content/RoadGuard_Artifacts/metrics

!cp /content/runs/road_damage/yolov8n_training/weights/best.pt /content/RoadGuard_Artifacts/models/
!cp /content/evaluation_metrics.json /content/RoadGuard_Artifacts/metrics/
!cp /content/runs/road_damage/yolov8n_training/results.png /content/RoadGuard_Artifacts/metrics/
!cp /content/runs/road_damage/yolov8n_training/confusion_matrix.png /content/RoadGuard_Artifacts/metrics/
!cp /content/runs/road_damage/yolov8n_training/results.csv /content/RoadGuard_Artifacts/metrics/

!cd /content/RoadGuard_Artifacts && zip -r /content/RoadGuard_Trained_Artifacts.zip .

# Copy to Google Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    !cp /content/RoadGuard_Trained_Artifacts.zip /content/drive/MyDrive/
    print("[+] Copied RoadGuard_Trained_Artifacts.zip to Google Drive!")

print("\n[+] Training and export complete!")
print("To deploy locally: extract RoadGuard_Trained_Artifacts.zip into your roadguard-ai/artifacts/ directory.")